In [1]:
import getpass
import pandas as pd
import requests
from bs4 import BeautifulSoup

password = getpass.getpass("Enter password")
schema = "wiki"
host = "127.0.0.1"
user = "root"
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

Enter password ········


In [3]:
# make everthing repeatable by creating a function
def get_city_info(city_name):
    url = "https://en.wikipedia.org/wiki/" + city_name.replace(" ", "_") # urls don't contain whitespace, use _ instead
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        country = soup.find("th", string="Country").find_next().get_text().strip()

        coordinates = soup.find("span", class_="geo-dec").get_text().strip().split()
        for coord in coordinates:
            num = float(coord[:-2])
            if coord.endswith("W") or coord.endswith("S"):
                num = -num
            if coord.endswith("N") or coord.endswith("S"):
                lat = num
            elif coord.endswith("E") or coord.endswith("W"):
                lon = num
        return city_name, country, lat, lon
    else:
        print(f"Trouble getting the HTML for {url}. Response code: {response.status_code}")
        return city_name, "N/A", None, None # if something went wrong, return "missing information"

In [ ]:
headers = {"User-Agent": "Mozilla/5.0"}
cities = ["Berlin", "Hamburg", "Munich"]

city_info = []

for city in cities:
    city_info.append(get_city_info(city))

city_info_df = pd.DataFrame(city_info, columns=["city", "country", "lat", "lon"])


In [ ]:
city_df = city_info_df[["city"]].copy()
city_df = city_df.rename(columns={"city": "city_name"})

In [ ]:
city_df.to_sql(
    "cities",
    con=connection_string,
    if_exists="append",
    index=False
)

In [49]:
cities_from_sql = pd.read_sql("SELECT * FROM cities", con=connection_string)

In [ ]:
city_info_with_id = city_info_df.merge(
    cities_from_sql,
    left_on="city",
    right_on="city_name"
)

In [ ]:
city_info_with_id = city_info_df.merge(
    cities_from_sql,
    left_on="city",
    right_on="city_name"
)

In [ ]:
city_info_sql_df = city_info_with_id[["city_id", "country", "lat", "lon"]].copy()

In [ ]:
city_info_sql_df.to_sql(
    "city_info",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
pd.read_sql("SELECT * FROM city_info", con=connection_string)

In [ ]:
#weather
cities_weather_df = pd.read_sql("""
SELECT c.city_id, c.city_name, ci.lat, ci.lon
FROM cities c
JOIN city_info ci
ON c.city_id = ci.city_id
""", con=connection_string)

In [4]:
lat = 52.52
lon = 13.405
api_key = ""
url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric"

In [ ]:
response = requests.get(url)
data = response.json()

In [2]:
#function for getting the weather from the api
def get_weather(lat, lon):
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        first = data["list"][0]
        
        temp = first["main"]["temp"]
        humidity = first["main"]["humidity"]
        visibility = first.get("visibility", None)
        
        return temp, humidity, visibility
    else:
        return None, None, None

In [ ]:
get_weather(52.52, 13.405)

In [12]:
weather_data = []

for i in range(len(cities_weather_df)):
    city_id = cities_weather_df.loc[i, "city_id"]
    lat = cities_weather_df.loc[i, "lat"]
    lon = cities_weather_df.loc[i, "lon"]
    
    temp, humidity, visibility = get_weather(lat, lon)
    
    weather_data.append((city_id, temp, humidity, visibility))

In [ ]:
weather_df = pd.DataFrame(
    weather_data,
    columns=["city_id", "temperature", "humidity", "visibility"]
)

In [ ]:
weather_df.to_sql(
    "weather",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
pd.read_sql("""
SELECT c.city_name, w.temperature, w.humidity, w.visibility
FROM cities c
JOIN weather w
ON c.city_id = w.city_id
""", con=connection_string)

In [ ]:
cities_df = pd.read_sql("SELECT * FROM cities", con=connection_string)

In [ ]:
import getpass
api_ninjas_key = getpass.getpass("")

In [22]:
#population function
def get_population(city):
    url = "https://api.api-ninjas.com/v1/city"
    headers = {"X-Api-Key": api_ninjas_key}
    
    querystring = {
        "name": city,
        "country": "DE"
    }
    
    city_request = requests.get(url, headers=headers, params=querystring)
    
    if city_request.status_code == 200:
        city_data = city_request.json()
        
        if len(city_data) > 0:
            return city_data[0]["population"]
        else:
            return None
    else:
        print(f"Could not get population for {city}")
        return None

In [23]:
from datetime import date

response_list = []

for i, city in cities_df.iterrows():
    population = get_population(city["city_name"])
    
    response_list.append((city["city_id"], population, date.today()))

In [ ]:
pop_df = pd.DataFrame(
    response_list,
    columns=["city_id", "population", "date_gathered"]
)

In [ ]:
pop_df.to_sql(
    "population",
    if_exists="append",
    con=connection_string,
    index=False
)

In [ ]:
pd.read_sql("SELECT * FROM population", con=connection_string)

In [ ]:
lat = 52.52
lon = 13.405

url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric"

response = requests.get(url)
data = response.json()

data['list'][0]

In [ ]:
first = data["list"][0]

forecast_time = first["dt_txt"]
outlook = first["weather"][0]["description"]
temperature = first["main"]["temp"]
feels_like = first["main"]["feels_like"]
wind_speed = first["wind"]["speed"]
rain_prob = first["pop"]

print(forecast_time, outlook, temperature, feels_like, wind_speed, rain_prob)

In [30]:
#forecast function
def get_forecast(lat, lon):
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        forecast_list = []
        
        for item in data["list"]:
            forecast_time = item["dt_txt"]
            outlook = item["weather"][0]["description"]
            temperature = item["main"]["temp"]
            feels_like = item["main"]["feels_like"]
            wind_speed = item["wind"]["speed"]
            rain_prob = item["pop"]
            
            forecast_list.append((
                forecast_time,
                outlook,
                temperature,
                feels_like,
                wind_speed,
                rain_prob
            ))
        
        return forecast_list
    else:
        return []

In [ ]:
get_forecast(52.52, 13.405)

In [38]:
#list for all forecasts
forecast_data = []

for i in range(len(cities_weather_df)):
    city_id = cities_weather_df.loc[i, "city_id"]
    lat = cities_weather_df.loc[i, "lat"]
    lon = cities_weather_df.loc[i, "lon"]
    
    city_forecast = get_forecast(lat, lon)
    
    for forecast_time, outlook, temperature, feels_like, wind_speed, rain_prob in city_forecast:
        forecast_data.append((
            city_id,
            forecast_time,
            outlook,
            temperature,
            feels_like,
            wind_speed,
            rain_prob
        ))

In [40]:
#new dataframe for forecasts

In [ ]:
forecast_df = pd.DataFrame(
    forecast_data,
    columns=[
        "city_id",
        "forecast_time",
        "outlook",
        "temperature",
        "feels_like",
        "wind_speed",
        "rain_prob"
    ]
)

forecast_df

In [ ]:
forecast_df.to_sql(
    "weather_forecast",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
#AIRPORTS

In [ ]:
#airports
aerodatabox_key = getpass.getpass("AeroDataBox key")

In [ ]:
lat = 52.52
lon = 13.405

url = f"https://aerodatabox.p.rapidapi.com/airports/search/location/{lat}/{lon}/km/50/16"

headers = {
    "X-RapidAPI-Key": aerodatabox_key,
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

querystring = {"withFlightInfoOnly": "true"}

response = requests.get(url, headers=headers, params=querystring)
response.json()

In [ ]:
response.json()["items"]

In [ ]:
pd.json_normalize(response.json()["items"])

In [17]:
#function for serching airports in one city
def get_airports(lat, lon):
    url = f"https://aerodatabox.p.rapidapi.com/airports/search/location/{lat}/{lon}/km/50/16"

    headers = {
        "X-RapidAPI-Key": aerodatabox_key,
        "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
    }

    querystring = {"withFlightInfoOnly": "true"}

    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["items"]

        airports = []

        for airport in data:
            icao = airport.get("icao")
            name = airport.get("name")
            timezone = airport.get("timeZone")

            airports.append((icao, name, timezone))

        return airports
    else:
        return []

In [ ]:
get_airports(52.52, 13.405)

In [19]:
icao = "EDDB"

In [23]:
from datetime import date, timedelta

tomorrow = date.today() + timedelta(days=1)

from_time_1 = f"{tomorrow}T00:00"
to_time_1 = f"{tomorrow}T11:59"

from_time_2 = f"{tomorrow}T12:00"
to_time_2 = f"{tomorrow}T23:59"

In [24]:
url_1 = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{from_time_1}/{to_time_1}"

querystring = {
    "withLeg": "true",
    "direction": "Arrival",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false"
}

headers = {
    "X-RapidAPI-Key": aerodatabox_key,
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

response_1 = requests.get(url_1, headers=headers, params=querystring)
data_1 = response_1.json()

In [ ]:
data_1.get("arrivals", [])[0]

In [ ]:
len(data_1.get("arrivals", []))

In [ ]:
#first_arrival
first_arrival = data_1.get("arrivals", [])[0]

origin_airport = first_arrival["departure"]["airport"]["name"]
scheduled_arrival = first_arrival["arrival"]["scheduledTime"]["local"]
terminal = first_arrival["arrival"].get("terminal")
gate = first_arrival["arrival"].get("gate")
airline = first_arrival["airline"]["name"]
flight_number = first_arrival["number"]
status = first_arrival["status"]

print(origin_airport, scheduled_arrival, terminal, gate, airline, flight_number, status)

In [25]:
#all arrivals time_1 loop
arrivals_list = []

for flight in data_1.get("arrivals", []):
    
    origin_airport = flight["departure"]["airport"]["name"]
    scheduled_arrival = flight["arrival"]["scheduledTime"]["local"]
    terminal = flight["arrival"].get("terminal")
    gate = flight["arrival"].get("gate")
    airline = flight["airline"]["name"]
    flight_number = flight["number"]
    status = flight["status"]
    
    arrivals_list.append((
        icao, 
        origin_airport,
        scheduled_arrival,
        terminal,
        gate,
        airline,
        flight_number,
        status
    ))

In [ ]:
# dataframe
arrivals_df = pd.DataFrame(
    arrivals_list,
    columns=[
        "icao",
        "origin_airport",
        "arrival_time",
        "terminal",
        "gate",
        "airline",
        "flight_number",
        "status"
    ]
)

In [27]:
url_2 = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{from_time_2}/{to_time_2}"

response_2 = requests.get(url_2, headers=headers, params=querystring)
data_2 = response_2.json()

In [28]:
#loop for second box
for flight in data_2.get("arrivals", []):
    
    origin_airport = flight["departure"]["airport"]["name"]
    scheduled_arrival = flight["arrival"]["scheduledTime"]["local"]
    terminal = flight["arrival"].get("terminal")
    gate = flight["arrival"].get("gate")
    airline = flight["airline"]["name"]
    flight_number = flight["number"]
    status = flight["status"]
    
    arrivals_list.append((
        icao,
        origin_airport,
        scheduled_arrival,
        terminal,
        gate,
        airline,
        flight_number,
        status
    ))

In [28]:
#arrival function
def get_arrivals_for_airport(icao):
    tomorrow = date.today() + timedelta(days=1)

    from_time_1 = f"{tomorrow}T00:00"
    to_time_1 = f"{tomorrow}T11:59"
    from_time_2 = f"{tomorrow}T12:00"
    to_time_2 = f"{tomorrow}T23:59"

    url_1 = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{from_time_1}/{to_time_1}"
    response_1 = requests.get(url_1, headers=headers, params=querystring)
    if response_1.status_code == 200:
        data_1 = response_1.json()
    else:
        data_1 = {"arrivals": []}

    url_2 = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{from_time_2}/{to_time_2}"
    response_2 = requests.get(url_2, headers=headers, params=querystring)
    if response_2.status_code == 200:
        data_2 = response_2.json()
    else:
        data_2 = {"arrivals": []}

    arrivals_list = []

    for flight in data_1.get("arrivals", []):
        arrivals_list.append((
            icao,
            flight["departure"]["airport"]["name"],
            flight["arrival"]["scheduledTime"]["local"],
            flight["arrival"].get("terminal"),
            flight["arrival"].get("gate"),
            flight["airline"]["name"],
            flight["number"],
            flight["status"]
        ))

    for flight in data_2.get("arrivals", []):
        arrivals_list.append((
            icao,
            flight["departure"]["airport"]["name"],
            flight["arrival"]["scheduledTime"]["local"],
            flight["arrival"].get("terminal"),
            flight["arrival"].get("gate"),
            flight["airline"]["name"],
            flight["number"],
            flight["status"]
        ))

    arrivals_df = pd.DataFrame(
        arrivals_list,
        columns=[
            "icao",
            "origin_airport",
            "arrival_time",
            "terminal",
            "gate",
            "airline",
            "flight_number",
            "status"
        ]
    )

    return arrivals_df

In [ ]:
get_arrivals_for_airport("EDDB").head()

In [ ]:
cities_airports_df = pd.read_sql("""
SELECT c.city_id, c.city_name, ci.lat, ci.lon
FROM cities c
JOIN city_info ci
ON c.city_id = ci.city_id
""", con=connection_string)

cities_airports_df.head()

In [ ]:
airports_df.head()

In [7]:
airports_data = []

for i in range(len(cities_airports_df)):
    city_id = cities_airports_df.loc[i, "city_id"]
    lat = cities_airports_df.loc[i, "lat"]
    lon = cities_airports_df.loc[i, "lon"]

    city_airports = get_airports(lat, lon)

    for icao, name, timezone in city_airports:
        airports_data.append((icao, name, timezone, city_id, True))

In [32]:
#loop for all airports
all_arrivals = []

for i in range(len(airports_df)):
    icao = airports_df.loc[i, "icao"]
    city_id = airports_df.loc[i, "city_id"]

    df = get_arrivals_for_airport(icao)

    if df is not None and not df.empty:
        df["city_id"] = city_id
        all_arrivals.append(df)

In [ ]:
len(all_arrivals)

In [ ]:
airports_df = pd.DataFrame(
    airports_data,
    columns=["icao", "name", "timezone", "city_id", "active"]
)
airports_df

In [ ]:
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
final_arrivals_df = pd.concat(all_arrivals, ignore_index=True)

In [35]:
flights_df = final_arrivals_df.copy()

In [ ]:
flights_df["arrival_time"] = pd.to_datetime(flights_df["arrival_time"])
flights_df["arrival_date"] = flights_df["arrival_time"].dt.date
flights_df["arrival_time_only"] = flights_df["arrival_time"].dt.time

In [37]:
flights_df = flights_df[
    [
        "icao",
        "origin_airport",
        "arrival_date",
        "arrival_time_only",
        "terminal",
        "gate",
        "airline",
        "flight_number",
        "status"
    ]
].copy()

In [38]:
flights_df = flights_df[
    [
        "icao",
        "origin_airport",
        "arrival_date",
        "arrival_time_only",
        "terminal",
        "gate",
        "airline",
        "flight_number",
        "status"
    ]
].copy()

In [ ]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

In [42]:
final_arrivals_df["arrival_time"] = final_arrivals_df["arrival_time"].str.replace(r"\+.*", "", regex=True)

In [ ]:
final_arrivals_df.to_sql(
    "arrivals",
    con=connection_string,
    if_exists="append",
    index=False
)